# LeetCode 2134 — Minimum Swaps to Group All 1's Together II bằng Python

Link đề: https://leetcode.com/problems/minimum-swaps-to-group-all-1s-together-ii/ · Độ khó: **Medium** · Chủ đề: Array, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def minSwaps(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến bài "số phép đổi chỗ tối thiểu" thành bài **cửa sổ trượt kích thước cố định**: chọn đoạn dài `k` (số số 1) chứa **nhiều số 1 nhất**; đáp án = `k − (số 1 trong đoạn)` = số số 0 trong đoạn.
2. Chứng minh vì sao số phép đổi chỗ đúng bằng số số 0 trong cửa sổ (cận dưới + cách đạt được) và kiểm chứng bằng **BFS trên không gian trạng thái**.
3. Xử lý **mảng vòng** bằng ba kỹ thuật: chỉ số `% n`, **nhân đôi mảng**, và **phần bù** (gom các số 0 trên mảng thẳng).
4. Dùng **tổng tiền tố** để đếm số 1 trong đoạn bất kỳ O(1).
5. Nắm các edge case: không có số 1, toàn số 1, `k = n`.

## 2. Đề bài & ví dụ

Một **phép đổi chỗ** (swap) = chọn hai vị trí khác nhau và đổi giá trị của chúng. Mảng **vòng**: phần tử đầu và cuối được coi là kề nhau.

Cho mảng nhị phân vòng `nums`, trả về **số phép đổi chỗ ít nhất** để mọi số `1` nằm liền nhau (ở vị trí nào cũng được, kể cả vắt qua cuối–đầu).

**Ví dụ 1:** `nums = [0,1,0,1,1,0,0]` → `1` (ví dụ `[0,0,1,1,1,0,0]`).

**Ví dụ 2:** `nums = [0,1,1,1,0,0,1,1,0]` → `2`.

**Ví dụ 3:** `nums = [1,1,0,0,1]` → `0` (các số 1 đã liền nhau nhờ tính vòng: vị trí 4, 0, 1).

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `nums[i] ∈ {0, 1}`.

In [1]:
from typing import List
from itertools import accumulate
from collections import deque
import random, time

examples = [
    ([0, 1, 0, 1, 1, 0, 0], 1),
    ([0, 1, 1, 1, 0, 0, 1, 1, 0], 2),
    ([1, 1, 0, 0, 1], 0),
]
for nums, exp in examples:
    print(nums, "số 1 =", nums.count(1), "-> kỳ vọng", exp)

[0, 1, 0, 1, 1, 0, 0] số 1 = 3 -> kỳ vọng 1
[0, 1, 1, 1, 0, 0, 1, 1, 0] số 1 = 5 -> kỳ vọng 2
[1, 1, 0, 0, 1] số 1 = 3 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Chỉ số vòng với `%`

Trên mảng vòng độ dài `n`, vị trí "sau" `n − 1` là `0`. Viết gọn: phần tử thứ `i` (với `i` có thể ≥ n) là `nums[i % n]`. Trong Python `%` luôn trả về số không âm khi `n > 0`, kể cả với `i` âm.

In [2]:
nums = [1, 1, 0, 0, 1]; n = len(nums)
print([nums[i % n] for i in range(3, 8)])     # đoạn vòng bắt đầu ở 3, dài 5
print(-1 % 5, -6 % 5)

[0, 1, 1, 1, 0]
4 4


### 3.2 Cửa sổ trượt kích thước cố định

Tổng của cửa sổ `[i, i + k)` chuyển sang `[i + 1, i + k + 1)` bằng cách **cộng phần tử vào** `nums[i + k]` và **trừ phần tử ra** `nums[i]` — O(1) mỗi bước thay vì O(k).

In [3]:
a = [0, 1, 0, 1, 1, 0, 0]; k = 3
s = sum(a[:k]); print("cửa sổ 0:", a[0:k], "tổng", s)
for i in range(1, len(a) - k + 1):
    s += a[i + k - 1] - a[i - 1]
    print(f"cửa sổ {i}:", a[i:i + k], "tổng", s, "(kiểm tra:", sum(a[i:i + k]), ")")

cửa sổ 0: [0, 1, 0] tổng 1
cửa sổ 1: [1, 0, 1] tổng 2 (kiểm tra: 2 )
cửa sổ 2: [0, 1, 1] tổng 2 (kiểm tra: 2 )
cửa sổ 3: [1, 1, 0] tổng 2 (kiểm tra: 2 )
cửa sổ 4: [1, 0, 0] tổng 1 (kiểm tra: 1 )


### 3.3 Tổng tiền tố

`pre[j] = nums[0] + … + nums[j − 1]`. Tổng đoạn `[l, r)` = `pre[r] − pre[l]`. Với mảng vòng, có thể dựng tiền tố trên mảng nhân đôi.

In [4]:
a = [0, 1, 1, 1, 0, 0, 1, 1, 0]
pre = list(accumulate(a, initial=0))
print("pre =", pre, "| tổng a[2:6] =", pre[6] - pre[2])

pre = [0, 0, 1, 2, 3, 3, 3, 4, 5, 5] | tổng a[2:6] = 2


### 3.4 BFS trên không gian trạng thái (để kiểm chứng)

Coi mỗi mảng là một **trạng thái**, mỗi phép đổi chỗ là một **cạnh**. Số phép tối thiểu = khoảng cách ngắn nhất từ trạng thái đầu tới một trạng thái "đã gom" ⇒ BFS. Chỉ khả thi với `n` nhỏ (số trạng thái ≤ C(n, k)), nhưng cho ta đáp án **chắc chắn đúng theo định nghĩa** để đối chiếu.

In [5]:
def grouped_circular(t):
    n = len(t); k = sum(t)
    if k in (0, n): return True
    # liền nhau trên vòng <=> số lần chuyển 0->1 khi đi vòng bằng 1
    return sum(1 for i in range(n) if t[i] == 1 and t[i - 1] == 0) == 1

for t in [(1, 1, 0, 0, 1), (0, 1, 0, 1, 1), (1, 1, 1), (0, 0, 1, 0)]:
    print(t, grouped_circular(t))

(1, 1, 0, 0, 1) True
(0, 1, 0, 1, 1) False
(1, 1, 1) True
(0, 0, 1, 0) True


## 4. Ý tưởng nền tảng / trực giác

Gọi `k` = số số 1. Sau khi gom, các số 1 chiếm một **đoạn vòng dài đúng `k`**. Hãy chọn trước đoạn đích `W`:
- Mỗi số **0** đang nằm trong `W` phải bị đổi ra ngoài.
- Mỗi số **1** đang nằm ngoài `W` phải được đổi vào.
- Hai số lượng này **bằng nhau** (vì `W` có đúng `k` ô và có đúng `k` số 1 tổng cộng), gọi là `z`.
- Mỗi phép đổi chỗ "một 0 trong `W`" ↔ "một 1 ngoài `W`" sửa được **cả hai** cùng lúc ⇒ cần đúng `z` phép.

⇒ Đáp án = `min` theo mọi đoạn vòng `W` dài `k` của **số số 0 trong `W`** = `k − max(số 1 trong W)`.

Tính tay ví dụ 1 (`k = 3`): các đoạn vòng dài 3:

| bắt đầu | đoạn | số 1 | số 0 = số phép |
|---|---|---|---|
| 0 | 0 1 0 | 1 | 2 |
| 1 | 1 0 1 | 2 | 1 |
| 2 | 0 1 1 | 2 | 1 |
| 3 | 1 1 0 | 2 | 1 |
| 4 | 1 0 0 | 1 | 2 |
| 5 | 0 0 0 | 0 | 3 |
| 6 | 0 0 1 (vòng) | 1 | 2 |

Min = 1. ✔

In [6]:
nums = [0, 1, 0, 1, 1, 0, 0]; n = len(nums); k = sum(nums)
for st in range(n):
    w = [nums[(st + t) % n] for t in range(k)]
    print(f"bắt đầu {st}: {w} -> số 0 = {w.count(0)}")

bắt đầu 0: [0, 1, 0] -> số 0 = 2
bắt đầu 1: [1, 0, 1] -> số 0 = 1
bắt đầu 2: [0, 1, 1] -> số 0 = 1
bắt đầu 3: [1, 1, 0] -> số 0 = 1
bắt đầu 4: [1, 0, 0] -> số 0 = 2
bắt đầu 5: [0, 0, 0] -> số 0 = 3
bắt đầu 6: [0, 0, 1] -> số 0 = 2


## 5. Cách 1 — Brute Force / baseline

### 5.1 BFS theo đúng định nghĩa (chuẩn vàng, n nhỏ)

In [7]:
def brute_bfs(nums):
    start = tuple(nums); n = len(nums)
    if grouped_circular(start): return 0
    dist = {start: 0}; q = deque([start])
    while q:
        s = q.popleft()
        for i in range(n):
            for j in range(i + 1, n):
                if s[i] != s[j]:                 # đổi hai giá trị bằng nhau là vô ích
                    t = list(s); t[i], t[j] = t[j], t[i]; t = tuple(t)
                    if t not in dist:
                        dist[t] = dist[s] + 1
                        if grouped_circular(t): return dist[t]
                        q.append(t)
    return 0

for nums, exp in examples:
    got = brute_bfs(nums)
    print(got, "OK" if got == exp else "SAI")

1 OK
2 OK
0 OK


**Độ phức tạp:** số trạng thái C(n, k) (tới ~10⁴ với n = 14) × O(n²) cạnh ⇒ chỉ dùng cho n ≤ 12.

### 5.2 Baseline: thử mọi đoạn vòng dài `k`, đếm lại từ đầu — O(n·k)

Dựa trên nhận xét ở mục 4 (chứng minh kỹ ở mục 10).

In [8]:
def brute_windows(nums):
    n = len(nums); k = sum(nums)
    if k == 0: return 0
    best = k
    for st in range(n):
        zeros = sum(1 for t in range(k) if nums[(st + t) % n] == 0)
        best = min(best, zeros)
    return best

for nums, exp in examples:
    print(brute_windows(nums), "OK" if brute_windows(nums) == exp else "SAI")

1 OK
2 OK
0 OK


**Độ phức tạp:** O(n·k), worst O(n²) (khi `k ≈ n/2`): 2,5·10⁹ với n = 10⁵ ⇒ TLE.

**Nút thắt:** hai cửa sổ liên tiếp chung `k − 1` phần tử, nhưng ta đếm lại từ đầu. ⇒ **cửa sổ trượt**.

In [9]:
for n in [2000, 6000]:
    a = [1] * (n // 2) + [0] * (n - n // 2); random.Random(1).shuffle(a)
    t = time.perf_counter(); brute_windows(a); print(f"n={n}: {(time.perf_counter()-t)*1000:.0f} ms")

n=2000: 105 ms


n=6000: 969 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cửa sổ trượt vòng):** duyệt `n` cửa sổ vòng dài `k` bắt đầu ở `0, 1, …, n − 1`. Khi trượt từ bắt đầu `s` sang `s + 1`: phần tử **vào** là `nums[(s + k) % n]`, phần tử **ra** là `nums[s]`. Giữ `cnt` = số 1 trong cửa sổ, cập nhật O(1).

**Insight 2 (nhân đôi):** đoạn vòng bắt đầu ở `s` dài `k` chính là đoạn thẳng `[s, s + k)` của mảng `nums + nums`. Dễ code nhưng tốn O(n) bộ nhớ thêm.

**Insight 3 (phần bù — không cần vòng):** một khối 1 trên vòng hoặc **không** vắt qua cuối–đầu (là đoạn thẳng), hoặc **có** vắt qua — khi đó các số **0** tạo thành một đoạn thẳng liền nhau ở giữa. Vậy đáp án = `min(` gom 1 trên mảng thẳng `,` gom 0 trên mảng thẳng `)`. (Gom 0 thành một khối thẳng cũng cần "số số 1 trong cửa sổ dài `n − k`" phép.)

**Quy tắc:** đáp án = `k − max_{cửa sổ vòng dài k}(số 1)`; nếu `k = 0` thì 0.

**Bất biến của vòng lặp trượt:** ngay sau khi cập nhật ở bước `s`, `cnt` = số số 1 trong cửa sổ vòng `{s, s+1, …, s+k−1} (mod n)` và `best` = max của `cnt` qua các cửa sổ bắt đầu ở `0..s`. Bất biến đúng ở `s = 0` (tính trực tiếp) và được giữ vì hai cửa sổ liên tiếp chỉ khác đúng một phần tử ở mỗi đầu.

**Một số trường hợp đặc biệt cần nghĩ trước:**
- `k = 0`: không có gì để gom ⇒ 0.
- `k = n`: chỉ có một cửa sổ (cả mảng) ⇒ 0 số 0 ⇒ 0.
- `k = 1`: mọi cửa sổ dài 1, cửa sổ chứa số 1 duy nhất có `cnt = 1` ⇒ 0.
- Mảng rất "xen kẽ" (`1010…`): mọi cửa sổ dài `k = n/2` có khoảng `k/2` số 1 ⇒ đáp án khoảng `n/4` — trường hợp tệ nhất về số swap.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cửa sổ trượt với chỉ số `% n`

In [10]:
def sliding_mod(nums):
    n = len(nums); k = sum(nums)
    if k == 0: return 0
    cnt = sum(nums[:k])          # cửa sổ bắt đầu ở 0
    best = cnt
    for s in range(1, n):        # cửa sổ bắt đầu ở s
        cnt += nums[(s + k - 1) % n] - nums[s - 1]
        best = max(best, cnt)
    return k - best

for nums, exp in examples:
    print(sliding_mod(nums), "OK" if sliding_mod(nums) == exp else "SAI")

1 OK
2 OK
0 OK


Cửa sổ bắt đầu ở `s` gồm `s, s+1, …, s+k−1` (mod n). So với cửa sổ bắt đầu ở `s − 1`: thêm `s + k − 1`, bớt `s − 1`.

Code nộp LeetCode (kiểu doocs: chạy chỉ số vào `i` từ `k` tới `n + k − 1`):

In [11]:
class SolutionMod:
    def minSwaps(self, nums: List[int]) -> int:
        k = nums.count(1)
        n = len(nums)
        mx = cnt = sum(nums[:k])
        for i in range(k, n + k):            # i = chỉ số vào (chưa mod)
            cnt += nums[i % n]
            cnt -= nums[(i - k) % n]         # chỉ số ra
            mx = max(mx, cnt)
        return k - mx

print([SolutionMod().minSwaps(x) for x, _ in examples])

[1, 2, 0]


Bản doocs duyệt `n` bước sau cửa sổ đầu (tổng cộng `n + 1` cửa sổ, cửa sổ cuối trùng cửa sổ đầu) — thừa một bước nhưng vô hại. Với `k = 0`: `sum(nums[:0]) = 0`, mỗi bước cộng và trừ cùng một phần tử ⇒ luôn 0 ⇒ trả 0. ✔

**Độ phức tạp cách 2:** O(n) thời gian, O(1) bộ nhớ.

### 7.2 Cách 3 — Nhân đôi mảng + tổng tiền tố

In [12]:
def doubled_prefix(nums):
    n = len(nums); k = sum(nums)
    if k == 0: return 0
    pre = list(accumulate(nums + nums, initial=0))
    best = max(pre[s + k] - pre[s] for s in range(n))
    return k - best

for nums, exp in examples:
    print(doubled_prefix(nums), "OK" if doubled_prefix(nums) == exp else "SAI")

1 OK
2 OK
0 OK


**Độ phức tạp cách 3:** O(n) thời gian, O(n) bộ nhớ (mảng nhân đôi + tiền tố). Ưu điểm: dễ đọc, dễ chứng minh.

### 7.3 Cách 4 — Phần bù: `min(gom 1 thẳng, gom 0 thẳng)` (không cần vòng)

Hàm `linear(target)`: số phép tối thiểu để gom mọi phần tử bằng `target` thành một khối **thẳng** = `m − max(số target trong cửa sổ thẳng dài m)`, với `m` = số phần tử bằng `target`.

In [13]:
def complement(nums):
    n = len(nums)
    def linear(target):
        m = sum(1 for x in nums if x == target)
        if m == 0: return 0
        cnt = sum(1 for x in nums[:m] if x == target); best = cnt
        for i in range(m, n):
            cnt += (nums[i] == target) - (nums[i - m] == target)
            best = max(best, cnt)
        return m - best
    return min(linear(1), linear(0))

for nums, exp in examples:
    print(complement(nums), "OK" if complement(nums) == exp else "SAI")

1 OK
2 OK
0 OK


Vì sao gom 0 thẳng cũng tốn đúng số phép như "gom 1 thành khối vòng vắt qua cuối–đầu"? Cả hai đều mô tả **cùng một cấu hình cuối**: 0 ở giữa thành khối liền ⇔ 1 ở hai đầu, liền nhau qua vòng. Số phép = số 1 lọt vào vùng đích của các số 0 = số 0 nằm ngoài vùng đó (vùng của các số 1).

**Độ phức tạp cách 4:** O(n), O(1) bộ nhớ.

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 2: `nums = [0,1,1,1,0,0,1,1,0]`, `n = 9`, `k = 5`.

| s (bắt đầu) | vào | ra | cnt | best |
|---|---|---|---|---|
| 0 | — | — | 0+1+1+1+0 = 3 | 3 |
| 1 | nums[5]=0 | nums[0]=0 | 3 | 3 |
| 2 | nums[6]=1 | nums[1]=1 | 3 | 3 |
| 3 | nums[7]=1 | nums[2]=1 | 3 | 3 |
| 4 | nums[8]=0 | nums[3]=1 | 2 | 3 |
| 5 | nums[0]=0 | nums[4]=0 | 2 | 3 |
| 6 | nums[1]=1 | nums[5]=0 | 3 | 3 |
| 7 | nums[2]=1 | nums[6]=1 | 3 | 3 |
| 8 | nums[3]=1 | nums[7]=1 | 3 | 3 |

Đáp án `5 − 3 = 2`. ✔

In [14]:
def trace_sliding(nums):
    n = len(nums); k = sum(nums)
    cnt = sum(nums[:k]); best = cnt
    print(f"n={n}, k={k}")
    print(f"s=0: cửa sổ {[nums[t % n] for t in range(k)]}, cnt={cnt}, best={best}")
    for s in range(1, n):
        inn, out = (s + k - 1) % n, s - 1
        cnt += nums[inn] - nums[out]; best = max(best, cnt)
        print(f"s={s}: vào nums[{inn}]={nums[inn]}, ra nums[{out}]={nums[out]} -> cửa sổ "
              f"{[nums[(s + t) % n] for t in range(k)]}, cnt={cnt}, best={best}")
    print("đáp án =", k - best)

trace_sliding([0, 1, 1, 1, 0, 0, 1, 1, 0])

n=9, k=5
s=0: cửa sổ [0, 1, 1, 1, 0], cnt=3, best=3
s=1: vào nums[5]=0, ra nums[0]=0 -> cửa sổ [1, 1, 1, 0, 0], cnt=3, best=3
s=2: vào nums[6]=1, ra nums[1]=1 -> cửa sổ [1, 1, 0, 0, 1], cnt=3, best=3
s=3: vào nums[7]=1, ra nums[2]=1 -> cửa sổ [1, 0, 0, 1, 1], cnt=3, best=3
s=4: vào nums[8]=0, ra nums[3]=1 -> cửa sổ [0, 0, 1, 1, 0], cnt=2, best=3
s=5: vào nums[0]=0, ra nums[4]=0 -> cửa sổ [0, 1, 1, 0, 0], cnt=2, best=3
s=6: vào nums[1]=1, ra nums[5]=0 -> cửa sổ [1, 1, 0, 0, 1], cnt=3, best=3
s=7: vào nums[2]=1, ra nums[6]=1 -> cửa sổ [1, 0, 0, 1, 1], cnt=3, best=3
s=8: vào nums[3]=1, ra nums[7]=1 -> cửa sổ [0, 0, 1, 1, 1], cnt=3, best=3
đáp án = 2


Trace cách 4 (phần bù) trên ví dụ 3 — khối 1 vắt qua vòng nên "gom 0 thẳng" cho 0:

In [15]:
nums = [1, 1, 0, 0, 1]
for target in [1, 0]:
    m = nums.count(target)
    wins = [(i, nums[i:i + m], nums[i:i + m].count(target)) for i in range(len(nums) - m + 1)]
    print(f"gom {target} thẳng: m={m}, các cửa sổ (bắt đầu, nội dung, số {target}) = {wins} -> {m - max(c for _, _, c in wins)}")

gom 1 thẳng: m=3, các cửa sổ (bắt đầu, nội dung, số 1) = [(0, [1, 1, 0], 2), (1, [1, 0, 0], 1), (2, [0, 0, 1], 1)] -> 1
gom 0 thẳng: m=2, các cửa sổ (bắt đầu, nội dung, số 0) = [(0, [1, 1], 0), (1, [1, 0], 1), (2, [0, 0], 2), (3, [0, 1], 1)] -> 0


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def minSwaps(self, nums: List[int]) -> int:
        n = len(nums)
        k = sum(nums)                    # số số 1 = độ dài cửa sổ
        if k == 0:
            return 0
        cnt = sum(nums[:k])              # số 1 trong cửa sổ bắt đầu ở 0
        best = cnt
        for s in range(1, n):            # cửa sổ vòng bắt đầu ở s
            cnt += nums[(s + k - 1) % n] - nums[s - 1]
            if cnt > best:
                best = cnt
        return k - best

In [17]:
sol = Solution()
for nums, exp in examples:
    got = sol.minSwaps(nums)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

1 | kỳ vọng 1 | OK
2 | kỳ vọng 2 | OK
0 | kỳ vọng 0 | OK


## 10. Vì sao thuật toán đúng

Gọi `k` = số số 1, `W` là một đoạn vòng dài `k`, `z(W)` = số số 0 trong `W` (= số số 1 ngoài `W`).

**Cận dưới:** xét lời giải bất kỳ, cấu hình cuối có các số 1 chiếm đúng một đoạn vòng `W*` dài `k`. Ban đầu có `z(W*)` số 1 nằm ngoài `W*`; mỗi phép đổi chỗ di chuyển **tối đa một** số 1 từ ngoài vào trong `W*` (một phép chỉ đụng hai ô). Vậy cần ≥ `z(W*)` ≥ `min_W z(W)` phép.

**Đạt được:** chọn `W` có `z(W)` nhỏ nhất. Ghép từng số 0 trong `W` với một số 1 ngoài `W` (hai tập cùng kích thước `z(W)`) và đổi chỗ từng cặp ⇒ đúng `z(W)` phép, sau đó `W` toàn số 1.

⇒ Đáp án = `min_W z(W) = k − max_W (số 1 trong W)`. Thuật toán duyệt đúng `n` đoạn vòng dài `k` (mọi điểm bắt đầu), cập nhật số 1 chính xác bằng bất biến "cnt = số 1 trong cửa sổ bắt đầu ở s". ∎

(Mục 12 kiểm chứng điều này với BFS trên mọi trạng thái cho n ≤ 10.)

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1a. BFS trạng thái | O(C(n,k)·n²) | O(C(n,k)) |
| 1b. Mọi cửa sổ, đếm lại | O(n·k) | O(1) |
| 2. Trượt với `% n` | O(n) | O(1) |
| 3. Nhân đôi + tiền tố | O(n) | O(n) |
| 4. Phần bù (thẳng) | O(n) | O(1) |

In [18]:
rng = random.Random(2134)
big = [rng.randint(0, 1) for _ in range(10**5)]
for name, f in [("sliding_mod", sliding_mod), ("SolutionMod", SolutionMod().minSwaps),
                ("doubled_prefix", doubled_prefix), ("complement", complement), ("Solution", Solution().minSwaps)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<15} -> {r}  {dt*1000:.1f} ms")

sliding_mod     -> 24868  9.5 ms
SolutionMod     -> 24868  9.7 ms
doubled_prefix  -> 24868  14.2 ms
complement      -> 24868  13.5 ms
Solution        -> 24868  7.7 ms


Đáp án lớn nhất có thể? Với `k` số 1, cửa sổ tốt nhất chứa ít nhất trung bình `k·k/n` số 1 (trung bình của mọi cửa sổ vòng đúng bằng `k²/n` vì mỗi số 1 thuộc đúng `k` cửa sổ) ⇒ đáp án ≤ `k − ⌈k²/n⌉`. Kiểm chứng nhanh:

In [19]:
rng = random.Random(11)
for _ in range(2000):
    n = rng.randint(1, 30); nums = [rng.randint(0, 1) for _ in range(n)]; k = sum(nums)
    if k:
        assert Solution().minSwaps(nums) <= k - (-(-k * k // n))
print("cận trên k - ceil(k^2/n) đúng trên 2000 mẫu")

cận trên k - ceil(k^2/n) đúng trên 2000 mẫu


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute_bfs": brute_bfs, "brute_windows": brute_windows, "sliding_mod": sliding_mod,
       "SolutionMod": SolutionMod().minSwaps, "doubled_prefix": doubled_prefix,
       "complement": complement, "Solution": Solution().minSwaps}
edge = examples + [
    ([0], 0), ([1], 0), ([0, 0, 0], 0), ([1, 1, 1], 0),
    ([1, 0, 1], 0),               # vòng: 2 -> 0 liền nhau
    ([1, 0, 1, 0], 1),
    ([1, 0, 0, 1, 0, 0, 1], 1),
    ([0, 1, 0, 1, 0, 1, 0, 1], 2),
]
for nums, exp in edge:
    for name, f in ALL.items():
        a = list(nums)
        assert f(a) == exp, (name, nums, f(list(nums)), exp)
        assert a == nums
print("Edge case OK:", len(edge))

Edge case OK: 11


### 12.2 Đối chiếu với BFS (chuẩn vàng) — n ≤ 10, seed cố định

In [21]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(1, 10)
    nums = [rng.randint(0, 1) for _ in range(n)]
    ref = brute_bfs(nums)
    for name, f in ALL.items():
        assert f(list(nums)) == ref, (name, nums)
print("400 case khớp BFS")

400 case khớp BFS


### 12.3 Đối chiếu các cách nhanh với baseline O(n·k) trên input vừa

In [22]:
rng = random.Random(7)
for _ in range(1500):
    n = rng.randint(1, 80); p = rng.random()
    nums = [1 if rng.random() < p else 0 for _ in range(n)]
    ref = brute_windows(nums)
    for name in ["sliding_mod", "SolutionMod", "doubled_prefix", "complement", "Solution"]:
        assert ALL[name](list(nums)) == ref, (name, nums)
print("1500 case vừa OK")

1500 case vừa OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên tính vòng (chỉ trượt trên mảng thẳng)

In [23]:
def wrong_linear_only(nums):
    n = len(nums); k = sum(nums)
    if k == 0: return 0
    cnt = sum(nums[:k]); best = cnt
    for i in range(k, n):
        cnt += nums[i] - nums[i - k]; best = max(best, cnt)
    return k - best

print("bản sai :", wrong_linear_only([1, 1, 0, 0, 1]))
print("bản đúng:", Solution().minSwaps([1, 1, 0, 0, 1]))

bản sai : 1
bản đúng: 0


**Vì sao sai:** bỏ qua các cửa sổ vắt qua cuối–đầu (ví dụ 3). Đây chính là bài LC 1151 (bản không vòng). **Sửa:** dùng `% n`, nhân đôi mảng, hoặc lấy `min` với "gom 0 thẳng".

### Lỗi 2 — Đếm số 1 **ngoài** cửa sổ nhưng chọn cửa sổ tệ (lấy max số 0)

In [24]:
def wrong_max_zero(nums):
    n = len(nums); k = sum(nums)
    if k == 0: return 0
    return max(sum(1 for t in range(k) if nums[(s + t) % n] == 0) for s in range(n))   # SAI: max

print("bản sai :", wrong_max_zero([0, 1, 0, 1, 1, 0, 0]))
print("bản đúng:", Solution().minSwaps([0, 1, 0, 1, 1, 0, 0]))

bản sai : 3
bản đúng: 1


**Sửa:** cần **min** số 0 (≡ **max** số 1) trong cửa sổ.

### Lỗi 3 — Nghĩ rằng mỗi số 1 ở xa phải "đi" nhiều bước (đếm khoảng cách)

In [25]:
def wrong_adjacent_swaps(nums):
    # SAI: tính số phép đổi chỗ KỀ NHAU (bubble) để gom về quanh trung vị - đây là bài khác
    pos = [i for i, x in enumerate(nums) if x == 1]
    k = len(pos)
    if k == 0: return 0
    q = [p - i for i, p in enumerate(pos)]
    med = q[k // 2]
    return sum(abs(x - med) for x in q)

print("bản sai :", wrong_adjacent_swaps([1, 0, 0, 0, 0, 1, 1]))
print("bản đúng:", Solution().minSwaps([1, 0, 0, 0, 0, 1, 1]))

bản sai : 4
bản đúng: 0


**Vì sao sai:** đề cho phép đổi chỗ **hai vị trí bất kỳ** (không cần kề nhau) ⇒ một phép đưa một số 1 từ bất kỳ đâu vào thẳng vị trí đích. Công thức khoảng cách/trung vị là của bài "đổi chỗ kề nhau" (LC 1703). Đọc kỹ định nghĩa "swap". (Thêm nữa, bản sai còn bỏ qua tính vòng.)

### Lỗi 4 — Chỉ số ra sai khi trượt (lệch một)

In [26]:
def wrong_out_index(nums):
    n = len(nums); k = sum(nums)
    if k == 0: return 0
    cnt = sum(nums[:k]); best = cnt
    for s in range(1, n):
        cnt += nums[(s + k - 1) % n] - nums[s]          # SAI: phải trừ nums[s-1]
        best = max(best, cnt)
    return k - best

print("ví dụ 1  - bản sai :", wrong_out_index([0, 1, 0, 1, 1, 0, 0]), "| bản đúng:", Solution().minSwaps([0, 1, 0, 1, 1, 0, 0]), "(trùng hợp đúng!)")
print("[0,0,1]  - bản sai :", wrong_out_index([0, 0, 1]), "| bản đúng:", Solution().minSwaps([0, 0, 1]))

ví dụ 1  - bản sai : 1 | bản đúng: 1 (trùng hợp đúng!)
[0,0,1]  - bản sai : 1 | bản đúng: 0


Bản sai vẫn qua ví dụ 1 của đề — lỗi lệch chỉ số thường "ẩn" kỹ như vậy. Với `[0,0,1]` (`k = 1`): ở `s = 2`, bản sai cộng `nums[2] = 1` rồi trừ `nums[2] = 1` ⇒ `cnt` không bao giờ đạt 1 ⇒ trả 1 thay vì 0.

**Sửa:** cửa sổ bắt đầu ở `s` không còn chứa `s − 1` ⇒ phần tử ra là `nums[s − 1]`. Mẹo: luôn in thử `cnt` và so với `sum(...)` tính lại cho vài bước đầu (như mục 3.2).

### Lỗi 5 — `k = 0` với cách nhân đôi dùng `max()` trên dãy rỗng / chia cho 0

In [27]:
def wrong_k_zero(nums):
    n = len(nums); k = sum(nums)
    pre = list(accumulate(nums + nums, initial=0))
    try:
        return k - max(pre[s + k] - pre[s] for s in range(n) if k > 0)   # rỗng khi k = 0
    except ValueError as e:
        return f"ValueError: {e}"

print("bản sai :", wrong_k_zero([0, 0, 0]))
print("bản đúng:", Solution().minSwaps([0, 0, 0]))

bản sai : ValueError: max() iterable argument is empty
bản đúng: 0


**Sửa:** xử lý `k == 0` riêng (trả 0) ngay đầu hàm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Gom tất cả X lại liền nhau với số **swap bất kỳ** ít nhất" ⇒ cửa sổ dài `count(X)` chứa nhiều X nhất ⇒ đáp án `count − max`.
- "Swap **kề nhau**" ⇒ bài khác: trung vị + khoảng cách (LC 1703, 2193).
- Mảng vòng: `% n`, nhân đôi, hoặc **phần bù** (khối vắt qua vòng ⇔ phần bù là khối thẳng) — kỹ thuật phần bù cũng dùng ở LC 918 Maximum Sum Circular Subarray, LC 1423.
- Cửa sổ cố định: tổng ban đầu + vòng lặp "cộng vào, trừ ra".
- Bài cùng họ: 1151 (bản không vòng), 1423 Maximum Points You Can Obtain from Cards, 2379.

## 15. Biến thể của bài

### 15.1 Biến thể A — Mảng **thẳng** (LC 1151)

In [28]:
def min_swaps_linear(data):
    k = sum(data)
    if k == 0: return 0
    cnt = sum(data[:k]); best = cnt
    for i in range(k, len(data)):
        cnt += data[i] - data[i - k]; best = max(best, cnt)
    return k - best

print(min_swaps_linear([1, 0, 1, 0, 1]), min_swaps_linear([0, 0, 0, 1, 0]), min_swaps_linear([1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1]))

1 0 3


### 15.2 Biến thể B — Trả về **vị trí bắt đầu** của khối đích (vòng) và **danh sách phép swap** cụ thể

In [29]:
def swaps_plan(nums):
    n = len(nums); k = sum(nums)
    if k == 0: return 0, []
    cnt = sum(nums[:k]); best, best_s = cnt, 0
    for s in range(1, n):
        cnt += nums[(s + k - 1) % n] - nums[s - 1]
        if cnt > best: best, best_s = cnt, s
    inside = {(best_s + t) % n for t in range(k)}
    zeros_in = [i for i in sorted(inside) if nums[i] == 0]
    ones_out = [i for i in range(n) if i not in inside and nums[i] == 1]
    return best_s, list(zip(zeros_in, ones_out))

for nums, _ in examples:
    st, plan = swaps_plan(nums)
    a = list(nums)
    for i, j in plan: a[i], a[j] = a[j], a[i]
    print(nums, "-> bắt đầu", st, "swap", plan, "->", a, "đã gom:", grouped_circular(tuple(a)))

[0, 1, 0, 1, 1, 0, 0] -> bắt đầu 1 swap [(2, 4)] -> [0, 1, 1, 1, 0, 0, 0] đã gom: True
[0, 1, 1, 1, 0, 0, 1, 1, 0] -> bắt đầu 0 swap [(0, 6), (4, 7)] -> [1, 1, 1, 1, 1, 0, 0, 0, 0] đã gom: True
[1, 1, 0, 0, 1] -> bắt đầu 4 swap [] -> [1, 1, 0, 0, 1] đã gom: True


### 15.3 Biến thể C — Gom các phần tử **bằng `x`** trong mảng nhiều giá trị (vòng)

Không còn chỉ 0/1, nhưng ý tưởng y hệt: cửa sổ dài `count(x)`, đếm số phần tử bằng `x`.

In [30]:
def min_swaps_value(nums, x):
    b = [1 if v == x else 0 for v in nums]
    return Solution().minSwaps(b)

print(min_swaps_value([3, 1, 3, 2, 3, 3, 1], 3), min_swaps_value([5, 5, 2, 5], 5))

1 0


### 15.4 Biến thể D — Gom **cả 1 lẫn 0** thành hai khối? (luôn đúng sau khi gom 1)

Trên mảng vòng nhị phân, khi các số 1 liền nhau thì các số 0 tự động liền nhau (phần bù của một cung là một cung). Nên "gom cả hai" có cùng đáp án — một quan sát nhỏ giải thích vì sao cách 4 (phần bù) đúng.

In [31]:
rng = random.Random(154)
for _ in range(300):
    a = tuple(rng.randint(0, 1) for _ in range(rng.randint(1, 12)))
    if grouped_circular(a):
        b = tuple(1 - x for x in a)
        assert grouped_circular(b)
print("1 liền nhau trên vòng => 0 liền nhau trên vòng: đúng trên 300 mẫu")

1 liền nhau trên vòng => 0 liền nhau trên vòng: đúng trên 300 mẫu


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. BFS | đúng định nghĩa | mũ | mũ | kiểm chứng n ≤ 10 |
| 1b. Mọi cửa sổ | đếm lại mỗi cửa sổ | O(n·k) | O(1) | baseline |
| 2. Trượt `% n` | cộng vào / trừ ra, chỉ số vòng | O(n) | O(1) | **mặc định** |
| 3. Nhân đôi + tiền tố | đoạn thẳng trên `nums+nums` | O(n) | O(n) | dễ đọc, nhiều truy vấn đoạn |
| 4. Phần bù | min(gom 1 thẳng, gom 0 thẳng) | O(n) | O(1) | tránh hẳn chỉ số vòng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minSwaps([1,0,1,0,1,0])` và `minSwaps([0,0,1,1,0,0,1])`?

**Bài 2 (tính tay):** Với `[1,0,0,1,1,0,1,0,0,1]` (`k = 5`), cửa sổ vòng nào chứa nhiều số 1 nhất? Đáp án là bao nhiêu?

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` cài cách 3 (nhân đôi + tiền tố).

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums)` cài cách 4 (phần bù), không dùng `%`.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums, queries)` trả lời nhiều truy vấn `(l, r)`: số swap tối thiểu để gom các số 1 **trong đoạn thẳng `nums[l..r]`** (coi đoạn là một mảng thẳng riêng, chỉ swap trong đoạn). Dùng một mảng tiền tố chung cho mọi truy vấn, mỗi truy vấn O(độ dài đoạn).

In [32]:
def bt3_todo(nums):
    # TODO: nhân đôi mảng + tổng tiền tố
    pass

def bt4_todo(nums):
    # TODO: min(gom 1 thẳng, gom 0 thẳng)
    pass

def bt5_todo(nums, queries):
    # TODO: mỗi truy vấn (l, r): gom số 1 trong nums[l..r] (thẳng)
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `k = 3` và `k = 3`. Mức 2: cửa sổ tốt nhất của `[1,0,1,0,1,0]` có 2 số 1 (ví dụ `0,1,2`); của `[0,0,1,1,0,0,1]` cũng có 2 (ví dụ `1,2,3` hoặc `2,3,4`; cửa sổ vòng `6,0,1` chỉ có 1). Mức 3: cả hai đều `3 − 2 = 1` — chạy `trace_sliding` để thấy từng cửa sổ.

**Bài 2.** Mức 1: liệt kê 10 cửa sổ. Mức 2: cửa sổ bắt đầu ở 9 (`9,0,1,2,3` = 1,1,0,0,1) có 3; bắt đầu ở 3 (`3..7` = 1,1,0,1,0) có 3. Mức 3: max = 3 ⇒ đáp án 2.

**Bài 3.** Mức 1: `pre = accumulate(nums + nums, initial=0)`. Mức 2: cửa sổ bắt đầu `s` có `pre[s+k] − pre[s]` số 1. Mức 3: `k − max(...)` với `s ∈ [0, n)`; nhớ `k = 0`.

**Bài 4.** Mức 1: viết hàm `linear(target)`. Mức 2: cửa sổ thẳng dài `m = count(target)`. Mức 3: `min(linear(1), linear(0))`.

**Bài 5.** Mức 1: mỗi truy vấn là bài 15.1 trên mảng con. Mức 2: `k = pre[r+1] − pre[l]`. Mức 3: số 1 trong cửa sổ `[i, i+k)` = `pre[i+k] − pre[i]`, duyệt `i` từ `l` tới `r − k + 1`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `1`, `1`. **Bài 2:** max số 1 trong cửa sổ = 3 ⇒ đáp án `2`.

In [33]:
assert Solution().minSwaps([1, 0, 1, 0, 1, 0]) == 1
assert Solution().minSwaps([0, 0, 1, 1, 0, 0, 1]) == 1
assert Solution().minSwaps([1, 0, 0, 1, 1, 0, 1, 0, 0, 1]) == 2

def bt3(nums):
    n = len(nums); k = sum(nums)
    if k == 0: return 0
    pre = list(accumulate(nums + nums, initial=0))
    return k - max(pre[s + k] - pre[s] for s in range(n))

def bt4(nums):
    n = len(nums)
    def linear(target):
        m = sum(1 for x in nums if x == target)
        if m == 0: return 0
        cnt = sum(1 for x in nums[:m] if x == target); best = cnt
        for i in range(m, n):
            cnt += (nums[i] == target) - (nums[i - m] == target); best = max(best, cnt)
        return m - best
    return min(linear(1), linear(0))

def bt5(nums, queries):
    pre = list(accumulate(nums, initial=0)); out = []
    for l, r in queries:
        k = pre[r + 1] - pre[l]
        if k == 0: out.append(0); continue
        best = max(pre[i + k] - pre[i] for i in range(l, r - k + 2))
        out.append(k - best)
    return out

rng = random.Random(1934)
for _ in range(1500):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 40))]
    ref = brute_windows(nums)
    assert bt3(nums) == ref and bt4(nums) == ref
    qs = []
    for _ in range(3):
        l = rng.randrange(len(nums)); r = rng.randrange(l, len(nums)); qs.append((l, r))
    assert bt5(nums, qs) == [min_swaps_linear(nums[l:r + 1]) for l, r in qs]
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
k = sum(nums); n = len(nums)
if k == 0: return 0
cnt = best = sum(nums[:k])
for s in range(1, n):                          # cửa sổ vòng bắt đầu ở s
    cnt += nums[(s + k - 1) % n] - nums[s - 1]
    best = max(best, cnt)
return k - best
```

**Checklist:**
- [ ] Độ dài cửa sổ = **số số 1**.
- [ ] Đáp án = `k − max(số 1 trong cửa sổ)` (= min số 0).
- [ ] Có xét cửa sổ **vắt qua vòng**.
- [ ] Chỉ số vào `s + k − 1`, chỉ số ra `s − 1`.
- [ ] `k = 0` trả 0; `k = n` trả 0.
- [ ] Swap **bất kỳ** (không phải kề nhau).